[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/069.%20The%20Shortest%20Path%20Problem/P69-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 69. The Shortest Path Problem

## Tier 3: Moth-Flame Optimization for a Constrained Freight Network

### Purpose and scope

This notebook applies a population-based metaheuristic, Moth-Flame Optimization (MFO), to find a low-cost route through a freight network. The input is a directed adjacency list of routes, with distance, congestion factors, and arrival-time windows. The output is a candidate path, its modeled cost, and diagnostic histories for the search.

The network is a fixed example, not a live or time-varying freight system. Congestion is represented by fixed multiplicative factors. Time windows are represented by a cost penalty, not as hard feasibility constraints. The model assumes travel begins at hour 0, uses an average speed of 60 miles per hour, and does not model waiting, service times, vehicle capacity, or changing traffic conditions. Costs are effective miles: distance multiplied by congestion and, when applicable, the time-window penalty.

MFO is stochastic and does not guarantee a globally optimal route. Its discrete path update is a heuristic adaptation of the logarithmic spiral movement used by MFO; it is not a continuous-coordinate optimization over node indices. Results can vary between runs because no random seed is set.

### Search outline

1. Generate an initial population of candidate paths using random, greedy, and partly randomized construction.
2. Evaluate each valid path using the distance, congestion, and arrival-time penalty model.
3. Rank candidates by fitness, where higher fitness means lower cost, and select the best candidates as flames.
4. Update other moths using a discrete, prefix-based adaptation of logarithmic spiral movement around a selected flame.
5. Replace invalid updated paths with a greedy candidate and periodically apply the notebook's local search.
6. Record best-cost and population-diversity diagnostics.

### Mathematical model

Let a path be \(P=(v_0,v_1,\ldots,v_k)\), where each consecutive pair is a directed edge. Let \(d_{ij}\) be its distance in miles and \(g_{ij}\geq 1\) its congestion factor. If the assumed speed is \(s=60\) miles per hour, arrival time at \(v_r\) is

$$
A_r = \left(\frac{\sum_{q=0}^{r-1} d_{v_qv_{q+1}}}{s}\right) \bmod 24,
\qquad r=1,\ldots,k.
$$

For a node with time window \([e_{v_r},\ell_{v_r}]\), define the penalty factor

$$
q_r =
\begin{cases}
1.5, & A_r < e_{v_r} \text{ or } A_r > \ell_{v_r},\\
1, & \text{otherwise}.
\end{cases}
$$

Nodes without a specified time window use \(q_r=1\). The path cost and fitness are

$$
C(P)=\sum_{r=1}^{k} d_{v_{r-1}v_r}\,g_{v_{r-1}v_r}\,q_r,
\qquad
F(P)=\frac{1}{1+C(P)}.
$$

Here, \(C(P)\) is the effective-mile objective minimized by the search; \(F(P)\) is the fitness maximized by MFO. A path must use existing directed edges. The time-window penalty makes this a path-dependent cost: the penalty on a node depends on the cumulative travel time along the route. The implementation evaluates this cumulative time using base distances, then applies congestion and the arrival penalty to the cost of each edge.

The number of possible simple paths can grow exponentially with network size. MFO samples and updates a population rather than enumerating all paths, so it provides a heuristic candidate rather than an optimality certificate. In this implementation, pairwise diversity calculation is quadratic in population size and also depends on path length.

### Interpreting the diagnostics

A decreasing best-cost history indicates that the recorded population has found lower-cost candidates. Diversity is the mean pairwise difference between valid paths, based on one minus the Jaccard similarity of their node sets. Low diversity means the population's paths use similar sets of nodes; it does not prove that the search has converged to a global optimum.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np

print("Libraries imported successfully")
print(f"NumPy version: {np.__version__}")

In [ ]:
@dataclass
class ComplexNetwork:
    """A freight network with edge costs and node arrival-time windows."""

    nodes: Dict[str, Tuple[float, float]]
    edges: Dict[str, Dict[str, float]]
    congestion: Dict[str, Dict[str, float]]
    time_windows: Dict[str, Tuple[int, int]]
    node_names: Dict[str, str]


@dataclass
class Moth:
    """A candidate route and its fitness in the MFO population."""

    position: List[str]
    fitness: float
    valid: bool


@dataclass
class MFOResult:
    """The best candidate and search diagnostics returned by MFO."""

    best_path: List[str]
    best_cost: float
    convergence_history: List[float]
    diversity_history: List[float]
    computation_time: float
    iterations: int

print("Data structures defined")

In [ ]:
def create_complex_freight_network() -> ComplexNetwork:
    """Create the example freight network and its model parameters."""
    nodes = {
        "LA": (0, 0),
        "PHX": (2, 1),
        "ALB": (3, 2),
        "DEN": (1, 2),
        "SALT": (2, 3),
        "KCM": (4, 2),
        "DAL": (3, 1),
        "OKC": (4, 1.5),
        "CHI": (5, 2),
        "DET": (5.5, 2.5),
        "MIN": (4.5, 3),
        "STL": (5, 1.5),
        "IND": (6, 2),
        "COL": (6.5, 2.5),
        "PIT": (7, 2),
        "ATL": (6, 1),
        "NYC": (8, 2),
        "BOS": (9, 3),
        "WAS": (7.5, 1.5),
    }

    node_names = {
        "LA": "Los Angeles", "PHX": "Phoenix", "ALB": "Albuquerque",
        "DEN": "Denver", "SALT": "Salt Lake City", "KCM": "Kansas City",
        "DAL": "Dallas", "OKC": "Oklahoma City", "CHI": "Chicago",
        "DET": "Detroit", "MIN": "Minneapolis", "STL": "St. Louis",
        "IND": "Indianapolis", "COL": "Columbus", "PIT": "Pittsburgh",
        "ATL": "Atlanta", "NYC": "New York", "BOS": "Boston",
        "WAS": "Washington DC",
    }

    edges = {
        "LA": {"PHX": 370, "DEN": 1015},
        "PHX": {"LA": 370, "ALB": 270, "DEN": 586, "KCM": 1150},
        "ALB": {"PHX": 270, "DEN": 230, "KCM": 560, "SALT": 520},
        "DEN": {"LA": 1015, "PHX": 586, "ALB": 230, "SALT": 530, "KCM": 600, "MIN": 920},
        "SALT": {"DEN": 530, "ALB": 520, "MIN": 1300, "KCM": 980},
        "KCM": {"PHX": 1150, "ALB": 560, "DEN": 600, "SALT": 980, "OKC": 350, "STL": 250, "CHI": 515, "MIN": 398},
        "DAL": {"OKC": 200, "STL": 450, "ATL": 795},
        "OKC": {"DAL": 200, "KCM": 350, "STL": 500},
        "CHI": {"KCM": 515, "MIN": 409, "DET": 281, "STL": 300, "IND": 180},
        "DET": {"CHI": 281, "MIN": 527, "COL": 200},
        "MIN": {"DEN": 920, "KCM": 398, "SALT": 1300, "CHI": 409, "DET": 527},
        "STL": {"KCM": 250, "OKC": 500, "DAL": 450, "CHI": 300, "IND": 250},
        "IND": {"CHI": 180, "STL": 250, "COL": 180, "ATL": 430},
        "COL": {"DET": 200, "IND": 180, "PIT": 180},
        "PIT": {"COL": 180, "ATL": 680, "NYC": 370, "WAS": 250},
        "ATL": {"IND": 430, "PIT": 680, "WAS": 640, "NYC": 888},
        "NYC": {"PIT": 370, "WAS": 230, "BOS": 215},
        "BOS": {"NYC": 215},
        "WAS": {"PIT": 250, "ATL": 640, "NYC": 230},
    }

    congestion = {
        "LA": {"PHX": 1.2, "DEN": 1.0},
        "PHX": {"LA": 1.2, "ALB": 1.1, "DEN": 1.0, "KCM": 1.3},
        "ALB": {"PHX": 1.1, "DEN": 1.0, "KCM": 1.2, "SALT": 1.1},
        "DEN": {"LA": 1.0, "PHX": 1.0, "ALB": 1.0, "SALT": 1.1, "KCM": 1.1, "MIN": 1.2},
        "SALT": {"DEN": 1.1, "ALB": 1.1, "MIN": 1.3, "KCM": 1.2},
        "KCM": {"PHX": 1.3, "ALB": 1.2, "DEN": 1.1, "SALT": 1.2, "OKC": 1.0, "STL": 1.1, "CHI": 1.2, "MIN": 1.1},
        "DAL": {"OKC": 1.1, "STL": 1.2, "ATL": 1.3},
        "OKC": {"DAL": 1.1, "KCM": 1.0, "STL": 1.1},
        "CHI": {"KCM": 1.2, "MIN": 1.1, "DET": 1.0, "STL": 1.1, "IND": 1.0},
        "DET": {"CHI": 1.0, "MIN": 1.1, "COL": 1.0},
        "MIN": {"DEN": 1.2, "KCM": 1.1, "SALT": 1.3, "CHI": 1.1, "DET": 1.1},
        "STL": {"KCM": 1.1, "OKC": 1.1, "DAL": 1.2, "CHI": 1.1, "IND": 1.0},
        "IND": {"CHI": 1.0, "STL": 1.0, "COL": 1.0, "ATL": 1.1},
        "COL": {"DET": 1.0, "IND": 1.0, "PIT": 1.0},
        "PIT": {"COL": 1.0, "ATL": 1.2, "NYC": 1.3, "WAS": 1.1},
        "ATL": {"IND": 1.1, "PIT": 1.2, "WAS": 1.2, "NYC": 1.4},
        "NYC": {"PIT": 1.3, "WAS": 1.1, "BOS": 1.0},
        "BOS": {"NYC": 1.0},
        "WAS": {"PIT": 1.1, "ATL": 1.2, "NYC": 1.1},
    }

    time_windows = {
        "LA": (0, 24), "PHX": (0, 24), "ALB": (6, 22), "DEN": (0, 24),
        "SALT": (0, 24), "KCM": (0, 24), "DAL": (0, 24), "OKC": (0, 24),
        "CHI": (0, 24), "DET": (6, 22), "MIN": (0, 24), "STL": (0, 24),
        "IND": (0, 24), "COL": (6, 22), "PIT": (0, 24), "ATL": (0, 24),
        "NYC": (0, 24), "BOS": (6, 22), "WAS": (0, 24),
    }

    return ComplexNetwork(nodes, edges, congestion, time_windows, node_names)


complex_network = create_complex_freight_network()
print(f"Complex network created with {len(complex_network.nodes)} nodes")
print("Network includes congestion factors and time windows")
print(f"Total directed edges: {sum(len(neighbors) for neighbors in complex_network.edges.values())}")

## Cost evaluation and candidate-path construction

A path is valid when it contains at least one edge and every consecutive node pair is a directed edge in the network. The cost function returns infinity for an invalid path so that it cannot be mistaken for a competitive candidate. For a valid path, arrival time at each next node includes the distance of the edge just traveled. The arrival hour is measured modulo 24, and an arrival outside that node's stated window multiplies that edge's cost by 1.5.

The random constructor favors shorter outgoing edges but remains stochastic. The greedy constructor chooses the neighbor whose displayed coordinates are closest to the target; it is a construction heuristic, not a guarantee of minimum path cost. Both are used to seed or recover candidate routes within the MFO search.

In [ ]:
def calculate_path_cost(network: ComplexNetwork, path: List[str]) -> float:
    """Return effective-mile cost, including congestion and arrival penalties."""
    if len(path) < 2:
        return float("inf")

    cumulative_distance = 0.0
    total_cost = 0.0

    for from_node, to_node in zip(path, path[1:]):
        if from_node not in network.edges or to_node not in network.edges[from_node]:
            return float("inf")

        base_distance = network.edges[from_node][to_node]
        cumulative_distance += base_distance
        arrival_hour = (cumulative_distance / 60.0) % 24
        time_penalty = 1.0

        if to_node in network.time_windows:
            earliest, latest = network.time_windows[to_node]
            if arrival_hour < earliest or arrival_hour > latest:
                time_penalty = 1.5

        congestion_factor = network.congestion.get(from_node, {}).get(to_node, 1.0)
        total_cost += base_distance * congestion_factor * time_penalty

    return total_cost


def is_valid_path(network: ComplexNetwork, path: List[str]) -> bool:
    """Check that a path has edges and each edge exists in the directed network."""
    if len(path) < 2:
        return False

    return all(
        from_node in network.edges and to_node in network.edges[from_node]
        for from_node, to_node in zip(path, path[1:])
    )


def generate_random_path(network: ComplexNetwork, source: str, target: str) -> List[str]:
    """Generate a randomized, non-repeating route, favoring shorter edges."""
    if source not in network.nodes or target not in network.nodes:
        raise ValueError("Source and target must be nodes in the network.")

    path = [source]
    current = source
    visited = {source}

    while current != target:
        neighbors = [
            neighbor for neighbor in network.edges.get(current, {})
            if neighbor not in visited
        ]

        if not neighbors:
            return generate_greedy_path(network, source, target)

        weights = [1.0 / network.edges[current][neighbor] for neighbor in neighbors]
        next_node = random.choices(neighbors, weights=weights, k=1)[0]
        path.append(next_node)
        visited.add(next_node)
        current = next_node

        if len(path) > len(network.nodes):
            return generate_greedy_path(network, source, target)

    return path


def generate_greedy_path(network: ComplexNetwork, source: str, target: str) -> List[str]:
    """Construct a route by repeatedly choosing the neighbor closest to the target."""
    if source not in network.nodes or target not in network.nodes:
        raise ValueError("Source and target must be nodes in the network.")

    path = [source]
    current = source
    visited = {source}

    while current != target:
        neighbors = [
            neighbor for neighbor in network.edges.get(current, {})
            if neighbor not in visited
        ]
        if not neighbors:
            break

        target_x, target_y = network.nodes[target]
        best_neighbor = min(
            neighbors,
            key=lambda neighbor: np.hypot(
                target_x - network.nodes[neighbor][0],
                target_y - network.nodes[neighbor][1],
            ),
        )
        path.append(best_neighbor)
        visited.add(best_neighbor)
        current = best_neighbor

    return path

print("Path utility functions defined")

## MFO components and the discrete path representation

In the standard MFO metaphor, moths move through a continuous search space toward flames, which are the best solutions found so far. Here, a moth is instead a sequence of node identifiers, so a continuous spiral equation cannot be applied directly to its position. The notebook uses a discrete adaptation: at each shared sequence position, it probabilistically copies the flame's node or retains the moth's node. The probability is determined by the magnitude of the spiral term and is capped at 1.

This representation keeps the source and target behavior tied to the candidate sequences, but a copied prefix can create a non-edge or an incomplete route. Every updated route is therefore checked, and an invalid update is replaced with a greedy candidate. The spiral update is a heuristic adaptation for this representation; its output is not guaranteed to be valid or to improve cost.

In [ ]:
def initialize_moth_population(
    network: ComplexNetwork,
    source: str,
    target: str,
    population_size: int,
) -> List[Moth]:
    """Create a mixed population to combine route variety and greedy candidates."""
    if population_size < 1:
        raise ValueError("Population size must be at least 1.")

    moths = []
    for index in range(population_size):
        if index < population_size // 3:
            path = generate_random_path(network, source, target)
        elif index < 2 * population_size // 3:
            path = generate_greedy_path(network, source, target)
        else:
            path = generate_greedy_path(network, source, target)
            if len(path) > 3:
                mid_point = len(path) // 2
                candidates = [
                    node for node in network.edges.get(path[mid_point - 1], {})
                    if node not in path[:mid_point]
                    and node not in path[mid_point + 1:]
                ]
                if candidates:
                    path[mid_point] = random.choice(candidates)

        valid = is_valid_path(network, path) and path[-1] == target
        cost = calculate_path_cost(network, path) if valid else float("inf")
        fitness = 1.0 / (1.0 + cost) if valid else 0.0
        moths.append(Moth(path, fitness, valid))

    return moths


def logarithmic_spiral_update(
    moth_position: List[str],
    flame_position: List[str],
    b: float = 1.0,
    t: float = None,
) -> List[str]:
    """Apply a discrete prefix-copy adaptation of logarithmic spiral movement."""
    if t is None:
        t = random.uniform(-1, 1)

    new_position = []
    shared_length = min(len(moth_position), len(flame_position))
    spiral_factor = np.exp(b * t) * np.cos(2 * np.pi * t)
    copy_probability = min(1.0, abs(spiral_factor))

    for index in range(shared_length):
        if random.random() < copy_probability:
            new_position.append(flame_position[index])
        else:
            new_position.append(moth_position[index])

    if new_position and flame_position and new_position[-1] != flame_position[-1]:
        new_position.append(flame_position[-1])

    return new_position


def calculate_population_diversity(moths: List[Moth]) -> float:
    """Return mean pairwise Jaccard distance between valid candidate paths."""
    valid_moths = [moth for moth in moths if moth.valid]
    if len(valid_moths) < 2:
        return 0.0

    total_difference = 0.0
    comparisons = 0

    for first_index in range(len(valid_moths)):
        for second_index in range(first_index + 1, len(valid_moths)):
            first_nodes = set(valid_moths[first_index].position)
            second_nodes = set(valid_moths[second_index].position)
            union = first_nodes | second_nodes
            similarity = len(first_nodes & second_nodes) / len(union) if union else 0.0
            total_difference += 1.0 - similarity
            comparisons += 1

    return total_difference / comparisons

print("MFO algorithm components defined")

### Fitness, flame selection, and search recurrence

At iteration \(t\), the population is ranked by fitness. The best candidates become flames, and the number of flames decreases as the search proceeds. For a population size \(N\), the implementation uses

$$
N_f(t)=\max\left(1,\left\lfloor N\left(1-\frac{t}{T}\right)\right\rfloor\right),
$$

where \(T\) is the maximum number of iterations. Each non-flame moth is assigned a flame by its rank modulo \(N_f(t)\), then updated with `logarithmic_spiral_update`. Valid candidate costs are converted to fitness by \(F=1/(1+C)\), so sorting fitness in descending order is equivalent to sorting cost in ascending order.

The algorithm also applies `local_search_improvement` to the best valid candidate every tenth iteration. That local search tries 2-opt reversals of intermediate route segments and accepts only valid, lower-cost candidates. This is part of the implemented search flow; it does not make the overall stochastic search an exact optimization method.

In [ ]:
def moth_flame_optimization(
    network: ComplexNetwork,
    source: str,
    target: str,
    population_size: int = 30,
    max_iterations: int = 100,
    spiral_constant: float = 1.0,
) -> MFOResult:
    """Search for a low-cost path using the notebook's discrete MFO procedure."""
    if max_iterations < 1:
        raise ValueError("Maximum iterations must be at least 1.")

    start_time = time.time()
    moths = initialize_moth_population(network, source, target, population_size)
    if not any(moth.valid for moth in moths):
        raise ValueError("The initial population contains no valid source-to-target path.")

    convergence_history = []
    diversity_history = []
    print(f"Starting MFO with {population_size} moths, {max_iterations} iterations")

    for iteration in range(max_iterations):
        moths.sort(key=lambda moth: moth.fitness, reverse=True)
        best_cost = 1.0 / moths[0].fitness - 1.0 if moths[0].fitness > 0 else float("inf")
        convergence_history.append(best_cost)
        diversity = calculate_population_diversity(moths)
        diversity_history.append(diversity)

        if iteration % 20 == 0:
            print(f"Iteration {iteration}: Best cost = {best_cost:.1f}, Diversity = {diversity:.3f}")

        num_flames = max(1, int(population_size * (1 - iteration / max_iterations)))
        flames = moths[:num_flames]
        new_moths = []

        for index, moth in enumerate(moths):
            if index < num_flames:
                new_moths.append(moth)
                continue

            flame = flames[index % num_flames]
            new_position = logarithmic_spiral_update(
                moth.position, flame.position, spiral_constant
            )

            if not is_valid_path(network, new_position) or new_position[-1] != target:
                new_position = generate_greedy_path(network, source, target)

            valid = is_valid_path(network, new_position) and new_position[-1] == target
            cost = calculate_path_cost(network, new_position) if valid else float("inf")
            fitness = 1.0 / (1.0 + cost) if valid else 0.0
            new_moths.append(Moth(new_position, fitness, valid))

        moths = new_moths

        if iteration % 10 == 0:
            best_moth = max(moths, key=lambda moth: moth.fitness)
            if best_moth.valid:
                improved_path = local_search_improvement(network, best_moth.position)
                if is_valid_path(network, improved_path):
                    improved_cost = calculate_path_cost(network, improved_path)
                    old_cost = calculate_path_cost(network, best_moth.position)
                    if improved_cost < old_cost:
                        best_moth.position = improved_path
                        best_moth.fitness = 1.0 / (1.0 + improved_cost)

    best_moth = max(moths, key=lambda moth: moth.fitness)
    if not best_moth.valid:
        raise RuntimeError("The search ended without a valid candidate path.")

    best_cost = calculate_path_cost(network, best_moth.position)
    computation_time = time.time() - start_time

    return MFOResult(
        best_path=best_moth.position,
        best_cost=best_cost,
        convergence_history=convergence_history,
        diversity_history=diversity_history,
        computation_time=computation_time,
        iterations=max_iterations,
    )


def local_search_improvement(network: ComplexNetwork, path: List[str]) -> List[str]:
    """Try valid 2-opt reversals and retain the lowest-cost route found."""
    if len(path) < 3:
        return path

    best_path = path.copy()
    best_cost = calculate_path_cost(network, best_path)

    for first_index in range(1, len(path) - 2):
        for last_index in range(first_index + 1, len(path) - 1):
            candidate = (
                path[:first_index]
                + path[first_index:last_index + 1][::-1]
                + path[last_index + 1:]
            )
            if is_valid_path(network, candidate):
                candidate_cost = calculate_path_cost(network, candidate)
                if candidate_cost < best_cost:
                    best_path = candidate
                    best_cost = candidate_cost

    return best_path

print("MFO algorithm defined")

## Run the search

The source and target are fixed for this example. The population size, iteration limit, and spiral constant are the main search settings. They control the search process, not the underlying network or cost definition. The returned path is the best valid candidate retained by the run; the reported cost is recomputed from that path using the same function used to evaluate candidates.

In [ ]:
source = "LA"
target = "NYC"

mfo_result = moth_flame_optimization(
    complex_network,
    source,
    target,
    population_size=30,
    max_iterations=100,
    spiral_constant=1.0,
)

print("\n=== MOTH-FLAME OPTIMIZATION RESULTS ===")
print(f"Source: {source} ({complex_network.node_names[source]})")
print(f"Target: {target} ({complex_network.node_names[target]})")
print(f"Best path: {' -> '.join(mfo_result.best_path)}")
print(f"Best cost: {mfo_result.best_cost:.1f} effective miles")
print(f"Computation time: {mfo_result.computation_time:.2f} seconds")
print(f"Iterations: {mfo_result.iterations}")

print("\nPath details with congestion and arrival-time factors:")
cumulative_distance = 0.0
reported_total = 0.0
for from_node, to_node in zip(mfo_result.best_path, mfo_result.best_path[1:]):
    base_distance = complex_network.edges[from_node][to_node]
    cumulative_distance += base_distance
    arrival_hour = (cumulative_distance / 60.0) % 24
    earliest, latest = complex_network.time_windows.get(to_node, (0, 24))
    time_penalty = 1.5 if arrival_hour < earliest or arrival_hour > latest else 1.0
    congestion = complex_network.congestion.get(from_node, {}).get(to_node, 1.0)
    effective_cost = base_distance * congestion * time_penalty
    reported_total += effective_cost
    print(
        f"  {from_node} -> {to_node}: {base_distance:.0f} mi × "
        f"{congestion:.1f} congestion × {time_penalty:.1f} time factor "
        f"= {effective_cost:.1f} effective mi; arrival hour = {arrival_hour:.2f}"
    )

assert is_valid_path(complex_network, mfo_result.best_path)
assert mfo_result.best_path[0] == source and mfo_result.best_path[-1] == target
assert np.isclose(reported_total, mfo_result.best_cost)
print(f"\nTotal effective cost: {reported_total:.1f} miles")

## Visualize the candidate and search diagnostics

The network panel shows directed edges, with edge color indicating congestion category and thicker arrows identifying edges in the selected route. The convergence panel reports the best cost recorded at the start of each iteration; the final returned route can include a local-search improvement made after that iteration's history entry. The diversity panel reports the mean pairwise Jaccard distance among valid candidates. These plots describe this run and do not establish global optimality.

In [ ]:
def visualize_mfo_results(
    network: ComplexNetwork,
    result: MFOResult,
    source: str,
    target: str,
) -> None:
    """Plot the network route, convergence history, and population diversity."""
    fig, (ax_network, ax_cost, ax_diversity) = plt.subplots(
        1, 3, figsize=(19, 6)
    )

    route_edges = set(zip(result.best_path, result.best_path[1:]))
    congestion_labels = [(1.0, "Low (≤ 1.0)"), (1.1, "Moderate (1.0–1.2)"), (1.3, "High (> 1.2)")]
    congestion_colors = {"Low (≤ 1.0)": "green", "Moderate (1.0–1.2)": "orange", "High (> 1.2)": "red"}

    for from_node, neighbors in network.edges.items():
        for to_node in neighbors:
            factor = network.congestion.get(from_node, {}).get(to_node, 1.0)
            label = "Low (≤ 1.0)" if factor <= 1.0 else "Moderate (1.0–1.2)" if factor <= 1.2 else "High (> 1.2)"
            color = congestion_colors[label]
            selected = (from_node, to_node) in route_edges
            ax_network.annotate(
                "",
                xy=network.nodes[to_node],
                xytext=network.nodes[from_node],
                arrowprops={
                    "arrowstyle": "->",
                    "color": color,
                    "lw": 3.0 if selected else 1.0,
                    "alpha": 0.9 if selected else 0.25,
                },
            )

    for node_id, (x_coord, y_coord) in network.nodes.items():
        if node_id == source:
            color = "red"
        elif node_id == target:
            color = "blue"
        elif node_id in result.best_path:
            color = "gold"
        else:
            color = "lightgreen"
        ax_network.scatter(
            x_coord, y_coord, s=350 if node_id in result.best_path else 250,
            c=color, edgecolors="black", linewidth=1.2, zorder=5,
        )
        ax_network.text(x_coord, y_coord - 0.2, node_id, fontsize=8, ha="center")

    ax_network.set_title("Freight Network and Selected Route")
    ax_network.set_xlabel("Schematic x-coordinate (not miles)")
    ax_network.set_ylabel("Schematic y-coordinate (not miles)")
    ax_network.set_aspect("equal")
    ax_network.grid(True, alpha=0.3)
    congestion_handles = [
        plt.Line2D([0], [0], color=congestion_colors[label], lw=2, label=label)
        for _, label in congestion_labels
    ]
    route_handle = plt.Line2D([0], [0], color="black", lw=3, label="Selected route")
    ax_network.legend(handles=congestion_handles + [route_handle], fontsize=8)

    iterations = range(len(result.convergence_history))
    ax_cost.plot(iterations, result.convergence_history, color="blue", label="Best recorded cost")
    ax_cost.axhline(result.best_cost, color="black", linestyle="--", label="Returned path cost")
    ax_cost.set_title("Best-Cost History")
    ax_cost.set_xlabel("Iteration")
    ax_cost.set_ylabel("Effective cost (miles)")
    ax_cost.grid(True, alpha=0.3)
    ax_cost.legend()

    ax_diversity.plot(iterations, result.diversity_history, color="green", label="Mean pairwise Jaccard distance")
    ax_diversity.set_title("Population Diversity")
    ax_diversity.set_xlabel("Iteration")
    ax_diversity.set_ylabel("Diversity (0–1)")
    ax_diversity.set_ylim(0, 1)
    ax_diversity.grid(True, alpha=0.3)
    ax_diversity.legend()

    fig.tight_layout()
    plt.show()


visualize_mfo_results(complex_network, mfo_result, source, target)

## Parameter sensitivity

This experiment varies population size and the spiral constant while keeping the network and source–target pair fixed. Because the search is stochastic and no random seed is set, an individual run cannot isolate parameter effects from random variation. Treat these runs as illustrative observations, not as a statistically controlled comparison. The final diversity value is calculated from the last population recorded in the search history.

In [ ]:
def parameter_sensitivity_analysis(
    network: ComplexNetwork,
    source: str,
    target: str,
) -> Tuple[List[dict], List[dict]]:
    """Show how selected MFO settings relate to single-run outcomes."""
    print("=== PARAMETER SENSITIVITY ANALYSIS ===")

    population_sizes = [10, 20, 30, 40, 50]
    population_results = []
    print("\nTesting different population sizes:")
    for population_size in population_sizes:
        result = moth_flame_optimization(
            network,
            source,
            target,
            population_size=population_size,
            max_iterations=50,
            spiral_constant=1.0,
        )
        final_diversity = result.diversity_history[-1] if result.diversity_history else 0.0
        population_results.append({
            "population_size": population_size,
            "best_cost": result.best_cost,
            "computation_time": result.computation_time,
            "final_diversity": final_diversity,
        })
        print(
            f"  Population {population_size:2d}: Cost = {result.best_cost:6.1f}, "
            f"Time = {result.computation_time:5.2f}s, Diversity = {final_diversity:.3f}"
        )

    spiral_constants = [0.5, 1.0, 1.5, 2.0]
    spiral_results = []
    print("\nTesting different spiral constants:")
    for spiral_constant in spiral_constants:
        result = moth_flame_optimization(
            network,
            source,
            target,
            population_size=30,
            max_iterations=50,
            spiral_constant=spiral_constant,
        )
        convergence_rate = (
            (result.convergence_history[0] - result.convergence_history[-1])
            / len(result.convergence_history)
            if len(result.convergence_history) > 1
            else 0.0
        )
        spiral_results.append({
            "spiral_constant": spiral_constant,
            "best_cost": result.best_cost,
            "computation_time": result.computation_time,
            "convergence_rate": convergence_rate,
        })
        print(
            f"  Spiral {spiral_constant:3.1f}: Cost = {result.best_cost:6.1f}, "
            f"Time = {result.computation_time:5.2f}s, Rate = {convergence_rate:.1f}"
        )

    return population_results, spiral_results


pop_results, spiral_results = parameter_sensitivity_analysis(
    complex_network, source, target
)

## Validate and interpret the returned route

The following checks verify that the selected route starts and ends at the requested nodes, uses existing edges, and has a cost consistent with the path-cost function. These checks establish internal consistency, not global optimality. The displayed node coordinates are schematic; route distances come from the edge data, not Euclidean distances on the plot.

In [ ]:
def summarize_selected_route(
    network: ComplexNetwork,
    result: MFOResult,
    source: str,
    target: str,
) -> None:
    """Check route feasibility and report its modeled cost components."""
    path = result.best_path
    assert path[0] == source
    assert path[-1] == target
    assert is_valid_path(network, path)

    cumulative_distance = 0.0
    component_total = 0.0
    for from_node, to_node in zip(path, path[1:]):
        distance = network.edges[from_node][to_node]
        cumulative_distance += distance
        arrival_hour = (cumulative_distance / 60.0) % 24
        earliest, latest = network.time_windows.get(to_node, (0, 24))
        time_factor = 1.5 if arrival_hour < earliest or arrival_hour > latest else 1.0
        congestion_factor = network.congestion.get(from_node, {}).get(to_node, 1.0)
        component_total += distance * congestion_factor * time_factor

    calculated_cost = calculate_path_cost(network, path)
    assert np.isclose(component_total, calculated_cost)
    assert np.isclose(calculated_cost, result.best_cost)

    print("=== ROUTE CONSISTENCY CHECK ===")
    print(f"Valid directed path: {'Yes' if is_valid_path(network, path) else 'No'}")
    print(f"Source and target match: {path[0]} -> {path[-1]}")
    print(f"Recomputed effective cost: {calculated_cost:.1f} miles")
    print(f"Reported effective cost: {result.best_cost:.1f} miles")
    print("The route is feasible under the edge-connectivity check and internally cost-consistent.")
    print("MFO does not provide a certificate that this route is globally optimal.")


summarize_selected_route(complex_network, mfo_result, source, target)

## Key takeaways

- The route objective combines base distance, fixed congestion multipliers, and a penalty for arriving outside specified time windows.
- Arrival time is computed from cumulative base distance at an assumed 60 miles per hour; congestion changes modeled cost, not modeled travel time.
- MFO ranks a population of paths by inverse cost, selects flames, and uses a discrete spiral-inspired update to create candidate routes.
- Invalid path updates are checked and replaced by a greedy candidate; periodic 2-opt local search can improve a valid route.
- Convergence and diversity plots describe the search behavior of a run. Neither is proof of global optimality.
- Results can vary between runs. The sensitivity experiment uses individual stochastic runs, so its outcomes should be interpreted cautiously.

### Natural extensions

The model could be extended with more realistic travel-time data, service durations, hard time-window feasibility, or additional operational constraints. Such extensions require corresponding changes to the path evaluation and candidate feasibility checks.